***

# Question 2: Climate Modelling

## Supplementary Jupyter Notebook 2 for Mathematical Modelling for Sustainable Development Coursework

***

Methods and assumptions are explained/modelled throughout. Inline comments are left to show how I coded this logically and not meant to be explanative - the markdown blocks serve this purpose!


Full development of this code can be found in the following link: 
[Link to Github Repo](https://github.com/Leonie-G-B/MathModSusDev)


> Student Num. 2101377

> Email: ch21886@bristol.ac.uk

***

### Nomenclature and utility functions: 

In [ ]:
# Simulation and modelling packages

import sympy as sym
from scipy.integrate import solve_ivp
from scipy.optimize import minimize 
from scipy.interpolate import interp1d


# Other required packages

import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt

# Utility funcitons: 

def decimal_year_to_datetime(y):
    year = int(y)
    remainder = y - year
    start = pd.Timestamp(f"{year}-01-01")
    end = pd.Timestamp(f"{year+1}-01-01")
    return start + (end - start) * remainder


***

## Brief task and code overview: 

This notebook shows the model and numbers that support the supplementary report. The aim to create a model that can help predict the future melting of the Greenland icesheet. This is done by using historical data to tune the model parameters, identify tipping points, and subsequently answer the key question: will the Greenland ice sheet melt completely? We take the expected global warming temperature deltas as indicated by the Representative Concentration Pathways (RCPs) leading up to 2100. 

### Primary equation we are using to model this 'system'

This is a representative equation, where (as you will see in resultant code blocks) the parameters are mix between estimated realistic values and optimised values using historical data.

$\huge   dh/dt=P- \frac{r(T_{0} + \Delta T(t) - T_{m})^2}{h}  - Fh$

where: 

$ h = $ height of the Greenland ice cap (average, [m])

$ T_{0} = $ the surface temperature oat the base of the ice cap, pre-industrial, positive (°C )

$ h(f) = $ characteristic length scale related to the altitude above which the ice will not melt ( $T = 0$ )

$ P = $ Precipitation rate ([m/year])

$ r = $ Melting rate parameter 

$ \delta T(t) = $ Temperature increase at a time t, due to global warming [°C]

$ T_{m} = $ Melting temperature of ice (°C)

$ F = $ Rate of flow of the ice sheet into the sea


So the first task here is to load in the historical data we're going to use, which can be found here: 

[Antarctic and Greenland Ice Sheet mass balance 1992-2020 for IPCC AR6](https://ramadda.data.bas.ac.uk/repository/entry/show?entryid=77b64c55-7166-4a06-9def-2e400398e452)



In [ ]:
gld_mass_balance_filename = 'Datasets/imbie_greenland_2021_Gt.csv' #relative filepath
gld_mass_balance_csv = pd.read_csv(gld_mass_balance_filename)


gld_mass_balance_csv["Day"] = (
    gld_mass_balance_csv["Year"]
    .astype(float)
    .apply(decimal_year_to_datetime)
) # converting data into useful format 

# plot for reference!

plt.figure(figsize=(10,5))
plt.plot(gld_mass_balance_csv["Day"], gld_mass_balance_csv["Cumulative mass balance (Gt)"])
plt.xlabel("Year"); plt.ylabel("Mass balance change from initial (giga tonnes)")
plt.title("Greenland Ice Sheet Mass Balance Change")
plt.grid()
plt.show()

***

## Time to introduce some assumptions 

1. The icesheet is a prism with a constant cross section: 

    $ V = w * l * h $

    therefore: $ V \propto h $

2. Ice sheet has a constant density: 

    $ m = \rho V $

    therefore: $ m \propto V $ and thus $ m \propto h $


This assumption allows us to use the above data to represent the change in height of the ice sheet, with the help of a datum value for calibration. 

[Mean thickness number](https://www.bbc.co.uk/news/science-environment-42260580)

Where the average thickness was stated to be roughly 1,673m in 2017 (this was derived from elevation maps of the actual land/bedrock and the ice sheet&snow ontop).


In [ ]:
calb_year = pd.to_datetime('2017-01-01')
calb_avg_thickness = 1673  # m

rho_ice = 917        # kg/m^3
sheet_area = 1.7e12  # m^2

date_idx_v2 = gld_mass_balance_csv["Day"].sub(calb_year).abs().idxmin()
calb_mass_v2 = gld_mass_balance_csv.loc[date_idx_v2, "Cumulative mass balance (Gt)"]
mass_kg_v2 = gld_mass_balance_csv["Cumulative mass balance (Gt)"] * 1e12
delta_V_v2 = mass_kg_v2 / rho_ice
delta_h_v2 = delta_V_v2 / sheet_area
delta_h_calb_v2 = delta_h_v2 - delta_h_v2.iloc[date_idx_v2]

gld_mass_balance_csv["height calibrated"] = (
    calb_avg_thickness + delta_h_calb_v2
)

t_historical = gld_mass_balance_csv["Day"]
h_historical = gld_mass_balance_csv["height calibrated"]

plt.figure(figsize=(10,5))
plt.plot(
    gld_mass_balance_csv["Day"],
    gld_mass_balance_csv["height calibrated"]
)
plt.xlabel("Year"); plt.ylabel("Greenland Ice Sheet Height (Calibrated, m)")
plt.title("Calibrated change in average Greenland ice sheet height between 1992 and 2021")
plt.grid()
plt.show()


***

## Now introduce the model and optimise parameters to roughly follow the above trend 

This is a rough approximation and as you can see the resultant optimisation is overfitted. Lots of tweaking was required as the optimiser preferred to elimiate the melting rate parameter in favour of a simple linear distribution. This fit was accepted as it gives sensible results that shall be shown later on. 
Another thing to note (retrospective comment on this model) - its very slow to react to temperature changes, so the fact this appears like such a poor fit to the data is unsurprising. It working on a timescale of under 30 years, when in future plot we are looking at scales that are in an order of magnitude higher !


In [ ]:
#### Constant, estimated parameters (i.e. non-optimised)

Tm = 0 #fairly certain of this fact
T0 = 2 #degC, roughly!

#### Other important setup steps and parameters (inc. initial values)

dT, h = sym.symbols('dT, h') 

t0 = t_historical.iloc[0]
t_years = (t_historical - t0).dt.days.values / 365.25 #time span

h_data = h_historical.values
h0 = h_data[0]

#### Equationfunctions

def delta_T(t): # starting off with simple representation for change in temperature. This will be altered in future simulations
    return 0.5 + 0.05*t

delta_T_interp = interp1d(
    t_years,
    delta_T(t_years),
    kind='linear',
    fill_value="extrapolate"
)

def dh_dt(t, h,r, F, P ):

    h = max(h[0], 1.0) #enforcing a muin h so that the solver doesnt break with high gradients

    dT = float(delta_T_interp(t))
    T = T0+ dT

    melt = r * (T - Tm)**2 / h
    flow = F * h

    dhdt = P - melt - flow

    return [dhdt]


#### Objective and supplementary determination equations ]

def compute_P(r, F, h0, t0):
    dT0 = delta_T(t0)
    return (r * (T0 + dT0 - Tm)**2) / h0 + F*h0

def simulate_height(r, F, P , t_span, h0): # model predicted height

    sol = solve_ivp(
        lambda t, h: dh_dt(t, h, r, F, P),
        t_span=[t_span[0], t_span[-1]],
        y0=[h0],
        t_eval=t_span,
        max_step=0.25
    )
    return sol.y[0]


def objective(params):
    r, F, P = params

    if r < 0 or F < 0 or P < 0:
        return 1e20 #if number non-physical c, apply large penalty

    h_model = simulate_height(r, F,P,   t_years, h0)
    eps = 1e-6
    rel_error = (h_model - h_data) / (h_data + eps)


    print(f"Error: {np.sum(rel_error**2)}")
    # print(f"Sqrt error : {error_sqrt}")

    return np.sum(rel_error**2)


##### Optimisation 

initial_guess = [1.0, 1e-3, 1.5] #initial values for r, F, and P respectively
bounds = [(0, None), (0, None), (0, None)] # negatibe is non-physical 


result = minimize(objective, initial_guess, bounds=bounds, method='L-BFGS-B') #  minimise the objective function
r_opt, F_opt, P_opt = result.x
 
h_fit = simulate_height(r_opt, F_opt, P_opt, t_years, h0)


##### Plot results 


print("Optimised parameters:")
print(f"r = {r_opt}")
print(f"F = {F_opt}")
print(f"P = {P_opt}")

plt.figure(figsize=(10,5))
plt.plot(t_historical, h_data, label="Historical (calibrated)")
plt.plot(t_historical, h_fit, label="Model fit")
plt.xlabel("Year"); plt.ylabel("Ice Sheet Height (m)")
plt.title("Model optimisation against calibrated historical data.")

plt.legend()
plt.grid()
plt.show()

***

### Now for some plots that characterise this model

And to identify & visualise any tipping points.


In [ ]:
def dhdt_fixed_T(h, T, r, F, P):
    return P - r*(T - Tm)**2 / h - F*h

dT_tests = [1.0, 10.0, 20.0, 30.0]

h_vals = np.linspace(150, 4000, 500) #not starting at 0 as this is an asymptote and messes with the visualisation of the graph

plt.figure()

for dT_test in dT_tests:
    T_test = T0 + dT_test
    dh_vals = dhdt_fixed_T(h_vals, T_test, r_opt, F_opt, P_opt)
    plt.plot(h_vals, dh_vals, label=f"delta T = {dT_test:.1f}")

# plt.figure()
# plt.plot(h_vals, dh_vals)
plt.axhline(0, linestyle='--')
plt.xlabel("h")
plt.ylabel("dh/dt")
plt.title(f"Rate of change of height for fixed scalar temp. increases")
plt.legend()
plt.grid()
plt.show()

This plot identifies the model behaviour for a fixed change in temperature (i.e. not time dependent, just a scalar input to the model representing a total change in global temperature).
I've plotted this for a range of delta T values to identify the rough location where the plot does not cross the x-axis. This will give us an indicatation the region the tipping point will be in later on (for this plot, somewhere between 20 and 30 deg).

In [ ]:
T_vals = np.linspace(0, 25, 200)
h_eq_stable = []
h_eq_unstable = []

f_symb = (F_opt * h**2) - (P_opt * h) + r_opt*(T0 + dT - Tm)**2
solutions = sym.solve(f_symb, h )

h1_func = sym.lambdify(dT, solutions[0], 'numpy') #no loop required here
h2_func = sym.lambdify(dT, solutions[1], 'numpy')

h1_vals = h1_func(T_vals)
h2_vals = h2_func(T_vals)

D = sym.discriminant(f_symb, h)
T_crit_expr = sym.solve(D, dT)

print(T_crit_expr)
plt.axvline(T_crit_expr[1], linestyle='--', color='red')

plt.plot(T_vals, h1_vals, label="Branch 1")
plt.plot(T_vals, h2_vals, label="Branch 2")

plt.xlabel("Temperature change, scalar [deg C]")
plt.ylabel("Ice height, h [m]")
plt.title("Bifurcation diagram")
# plt.legend(loc='center left')
plt.grid()
plt.show()

This plot shows the behaviour of the ice sheet ('system state' - long term behaviout) against a set total increase in temperature ('control variable'). It clearly demonstrates a clear tipping point for the system - the upper line is a stable equilibria (should the temperture change reverse it will return back to this) and the lower an-unstable equilibria where the system 'collapses', i.e. the ice sheet completely melts and we would likely require some level of global freezing to return back to the previous stability point. 
We now know this occurs around 22.5 deg warming.

***

Now to plot this in a more meaningful and clear way - show the height of the ice cap over time with the increase in temperature distribution. Along with reference lines at this 'tipping point': 


In [ ]:
fig, ax1 = plt.subplots(figsize=(10,5))

t_span = np.linspace(0, 1350, 1000)
h0_values = [800, 1673, 3000]

delta_T_interp = interp1d( #as beofore but using a much longer time frame
    t_span,
    delta_T(t_span),
    kind='linear',
    fill_value="extrapolate"
)

# ax1.plot(t_years, h_fit, label="Ice height h(t)", color='blue')

for h0_i in h0_values:
    sol = solve_ivp(
        lambda t, h: dh_dt(t, h, r_opt, F_opt, P_opt),
        [t_span[0], t_span[-1]],
        [h0_i],
        t_eval=t_span,
        max_step=0.25
    )

    h_plot = np.maximum(sol.y[0], 0)

    plt.plot(t_span, h_plot, label=f"Ice sheet height, h0 = {h0_i}")

ax1.set_xlabel("Time (years)")
ax1.set_ylabel("Ice height (m)", color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

ax2 = ax1.twinx()

T_vals = T0 + delta_T_interp(t_span)

ax2.plot(t_span, T_vals, label="Temperature T(t)", linestyle='--', color='red')
ax2.set_ylabel("Temperature (°C)", color='red')
ax2.tick_params(axis='y', labelcolor='red')

ax2.axhline(T_crit_expr[1], color='black', linestyle=':', label="Tipping threshold (~22.5deg)")
diff = T_vals - T_crit_expr[1]
cross_idx = np.where(np.diff(np.sign(diff)) != 0)[0]
cross_idx = cross_idx[0]
t_cross = t_span[cross_idx] + (t_span[cross_idx+1] - t_span[cross_idx]) * (0 - diff[cross_idx]) / (diff[cross_idx+1] - diff[cross_idx])
ax2.axvline(t_cross, color='black', linestyle=':', label="Tipping threshold (time of occurance)")


lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)

plt.title("Transient response of the ice cap for varying initial heights and dynamic linear warming")
plt.grid()
plt.show()

This is a very useful plot that demonstrates all the behaviour we previously expected. Note how the lowest initial height line is increasing in height until it reaches that critical temperature value of ~22degC at which point it accelerates quickly towards 0. It's subtle but you can also see that for the highest initial height, it seems to be slightly levelling out up to the critical point (~400 years with this temperature distribution) before also accelerating towards 0. 

***

One final thing that will be useful to quote for the report is how this translates to sea level rise, before we move on to simulating the height of the ice cap with the predicted RCPs (see later on). This is simple due to the previous plots and assumptions for the mass balance change, and the fact that we know that the melting of the Greenland ice sheet has caused ~14mm of SL rise since 1992. 


In [ ]:
# Ice sheet height change since 1992 (from previous model fit data as this time frame works conventiently)
h_change = h_fit - h_fit[0]  # relative to 1992

#Scale factor from Greenland height change to global SL
delta_h_model_total = h_fit[-1] - h_fit[0]  # meters

SL_known = 0.014  # meters (14 mm) - given in brief and from original dataset 

# Conversion factor
scale_factor_SL_rise = SL_known / delta_h_model_total

# Corresponding sea level rise
SL_rise = h_change * scale_factor_SL_rise

print(f"Change in heihgt to SL change scale factor= {scale_factor_SL_rise}")
print(f"I.e. If the ice cap completely melted, it would produce a SL rise of:  {scale_factor_SL_rise*-1673} m ")

# Plot
fig, ax1 = plt.subplots(figsize=(10,5))

# Left axis: ice sheet height
ax1.plot(t_historical, h_change, 'b', label='Greenland Ice Sheet Height Change')
ax1.set_xlabel('Year')
ax1.set_ylabel('Change in Ice Sheet Height (m)', color='b')
ax1.tick_params(axis='y', labelcolor='b')
ax1.grid(True)

# Right axis: global sea-level rise
ax2 = ax1.twinx()
ax2.plot(t_historical, SL_rise, 'r', label='Global Sea Level Rise')
ax2.set_ylabel('Global Sea Level Rise (m)', color='r')
ax2.tick_params(axis='y', labelcolor='r')
ax2.invert_yaxis() 

# Title and legend
fig.suptitle('Ice Sheet Height Change vs Global Sea Level Rise')
fig.tight_layout()
plt.show()



As you can see from the printed information , this model when calibrated with the limited data (behaviour from 1992 to 2021) and extrapolated, the total SL rise slightly overestimates from the research predictions. But this is not massively concerning as mentioned there is a lot of uncertainty given we are extrapolating so much and assuming a constant density of the ice cap - this discrepancy isn't a concern.

In [ ]:

baseline_yr = 2005
end_year = 2100
T0_val = 0.91 # this is where the OWID graph diverges from, in 2005

T26_end = 1.7
T85_end = 4.0

#linearly fit thesse numbers 

m26, c26 = np.polyfit(np.array([baseline_yr, end_year]), np.array([T0_val, T26_end]), 1)
m85, c85 = np.polyfit(np.array([baseline_yr, end_year]), np.array([T0_val, T85_end]), 1)

print("RCP 2.6: ΔT(t) = {:.6f} * t + {:.6f}".format(m26, c26))
print("RCP 8.5: ΔT(t) = {:.6f} * t + {:.6f}".format(m85, c85))

years = np.linspace(baseline_yr, end_year, 200)

T26 = m26*years + c26
T85 = m85*years + c85

T26_low  = (0.9 -0.91)/(2100-2005) * (years - 2005)+0.91
T26_high = (2.3 -0.91)/(2100-2005) * (years - 2005)+0.91
T85_low  = (3.2 -0.91)/(2100-2005) * (years - 2005)+0.91
T85_high = (5.4 -0.91)/(2100-2005) * (years - 2005)+0.91

plt.figure(figsize=(10,5))

plt.plot(years, T26, label="RCP 2.6", color='blue')
plt.fill_between(years, T26_low, T26_high, alpha=0.3, color='blue')

plt.plot(years, T85, label="RCP 8.5", color='red')
plt.fill_between(years, T85_low, T85_high, alpha=0.3, color='red')

plt.scatter([2005], [0.91], color='black', label="2005 baseline")

plt.xlabel("Year")
plt.ylabel("Global Average Temperature Increase \n(relative to pre-industrial era) [°C]") #pre-industrial era is 1750
plt.title("Temperature Projections (Linearised RCP Scenarios 2.6 and 8.5)")
plt.legend()
plt.grid()
plt.show()


def delta_T_rcp26(t):
    year = 1992 + t
    return m26*year + c26

def delta_T_rcp85(t):
    year = 1992 + t
    return m85*year + c85

def dh_dt_scenario(t, h, r, F, P, delta_T_func):
    h_safe = max(h[0], 1.0)
    dT = delta_T_func(t)
    T = T0 + dT
    dhdt = P - r*(T - Tm)**2 / h_safe - F*h_safe
    return [dhdt]

def simulate(delta_T_func, t_span: list = [2017-1992, end_year-1992]):
    # t_span = [2017-1992, end_year-1992]
    t_eval = np.linspace(t_span[0], t_span[1], 500)

    sol = solve_ivp(
        lambda t, h: dh_dt_scenario(t, h, r_opt, F_opt, P_opt, delta_T_func),
        t_span,
        [1673],
        t_eval=t_eval
    )
    return sol.t, sol.y[0]

t26, h26 = simulate(delta_T_rcp26)
t85, h85 = simulate(delta_T_rcp85)


fig, ax1 = plt.subplots(figsize=(10,5))

ax1.plot(1992 + t26, h26, label="RCP 2.6")
ax1.plot(1992 + t85, h85, label="RCP 8.5")

ax1.set_xlabel("Year")
ax1.set_ylabel("Ice Sheet Height (m)")
ax1.grid()
ax1.legend(loc="lower left", fontsize = "large")
ax2 = ax1.twinx()

h26_SL = (h26 - h_fit[0]) * scale_factor_SL_rise
h85_SL = (h85 - h_fit[0]) * scale_factor_SL_rise

ax2.plot(1992 + t26, h26_SL, '--')
ax2.plot(1992 + t85, h85_SL, '--')

ax2.set_ylabel("Global Sea Level Rise (m)")
ax2.invert_yaxis()

plt.title("Greenland Ice Sheet Response to warming scenarios and resultant SL rise")
plt.show()

print("Total SL rises at 2100: ")
print(f"RCP 2.6: {h26_SL[-1]*1000} mm ")
print(f"RCP 8.5: {h85_SL[-1]*1000} mm ")


### Again but with a much larger timescale 

end_year = 3600

t26, h26 = simulate(delta_T_rcp26, [2017 - 1992, end_year - 1992])
t85, h85 = simulate(delta_T_rcp85, [2017 - 1992, end_year - 1992])


fig, ax1 = plt.subplots(figsize=(10,5))

ax1.plot(1992 + t26, h26, label="RCP 2.6")
ax1.plot(1992 + t85, h85, label="RCP 8.5")

ax1.set_xlabel("Year")
ax1.set_ylabel("Ice Sheet Height (m)")
ax1.grid()
ax1.legend(loc="lower left", fontsize = "large")
ax2 = ax1.twinx()

h26_SL = (h26 - h_fit[0]) * scale_factor_SL_rise
h85_SL = (h85 - h_fit[0]) * scale_factor_SL_rise

ax2.plot(1992 + t26, h26_SL, '--')
ax2.plot(1992 + t85, h85_SL, '--')

ax2.set_ylabel("Global Sea Level Rise (m)")
ax2.invert_yaxis()

plt.title("Greenland Ice Sheet Response to warming scenarios and resultant SL rise")
plt.show()

print(f"Total SL rises at {end_year}: ")
print(f"RCP 2.6: {h26_SL[-1]*1000} mm ")
print(f"RCP 8.5: {h85_SL[-1]*1000} mm ")

This gives a lot of info that can be used in the report to convey clearly the future of the earth's SL due to the melting of Greenland (with many caveats and assumptions to be made clear). Obviously the model here is simplified as I have taken a linear interpolation of the warming targets, but the reality is that the model doesnt respond in a notable way to small pertubations in the temperature over such small timescales, so this interpolation is not detrimental to the key results and outcomes!

In [ ]:
#one last plot that will be used in the report
t_span = np.linspace(2005, 2100, 500)
t_model = t_span - 1992

def simulate_rcp(delta_T_func):
    sol = solve_ivp(
        lambda t, h: dh_dt_scenario(t, h, r_opt, F_opt, P_opt, delta_T_func),
        [t_model[0], t_model[-1]],
        [1673],  # single initial condition
        t_eval=t_model,
        max_step=0.25
    )
    
    return np.maximum(sol.y[0], 0)

h26 = simulate_rcp(delta_T_rcp26)
h85 = simulate_rcp(delta_T_rcp85)

fig, ax1 = plt.subplots(figsize=(10,5))

ax1.plot(t_span, h26, label="Ice height (RCP 2.6)", color='blue')
ax1.plot(t_span, h85, label="Ice height (RCP 8.5)", color='red')

ax1.set_xlabel("Year")
ax1.set_ylabel("Ice height (m)", color='black')
ax1.grid(which = "both")

ax2 = ax1.twinx()

T26_vals = m26*t_span + c26
T85_vals = m85*t_span + c85

ax2.plot(t_span, T26_vals, '--', color='blue', label="Temperature (RCP 2.6)")
ax2.plot(t_span, T85_vals, '--', color='red', label="Temperature (RCP 8.5)")

ax2.set_ylabel("Temperature increase (°C)")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc = "center right")

plt.title("Transient ice sheet response under RCP temperature forcing (2005–2100)")
plt.show()

### Extension: Other tipping points discussion

As an extension, another tipping point to consider that influences the behaviour of the Greenland Ice sheet is the involvement and importance of warming ocean temperatures. There exists a balanced system of ocean currents all over the globe due to the formation of cold water currents from deep within the artic and antartic ocean - should temperate increases cause the integrity of these cold water sources to be jeopardised, the likely outcome would massively affect the ocean water temperatures. This would have a distinct impact on the 'flow' term in our model as the breaking off of ice ('caving') is worsened with warmer ocean temperatures. This is a particularly difficult system to model and the exact impact on the ideas presented in the report are discussed in a brief way to indicate that the optimistic model results as shown above are likely underrepresenting the other systems affected by increasing global temperatures (to name just one caveat of course). 
The source of this discussion can be found here: ["New Research Sparks Concerns That Ocean Circulation Will Collapse", Yale Environment 360](https://e360.yale.edu/features/climate-change-ocean-circulation-collapse-antarctica). This paints a damning picture of the state of the ocean currents in the not so distant future, suggesting that 'deep-water formation looks headed to collapse this century' which will affect many elements of our ecosystem, not just weather patterns and ice cap melting. Note however that this report is taking the worst case scenario for global warming, i.e. assuming we maintain the current CO2 emissions and general global warming trends, when in fact we likely have already surpassed the peak CO2 emissions as a globe. 